In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import time
 
# Muat dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
 
# Normalisasi dan tambahkan channel dimension
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = x_train[..., tf.newaxis]
x_test = x_test[..., tf.newaxis]
 
# Dataset pipeline
batch_size = 64
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(buffer_size=1024).batch(batch_size)
val_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(batch_size)

In [ ]:
class MyModel(tf.keras.Model):
  def __init__(self):
    super(MyModel, self).__init__()
    self.flatten = tf.keras.layers.Flatten()
    self.dense1 = tf.keras.layers.Dense(128, activation='relu')
    self.dense2 = tf.keras.layers.Dense(10)
 
  def call(self, x):
    x = self.flatten(x)
    x = self.dense1(x)
    return self.dense2(x)

In [ ]:
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = tf.keras.optimizers.Adam()
 
train_loss_metric = tf.keras.metrics.Mean(name='train_loss')
train_accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
val_accuracy_metric = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')

In [ ]:
@tf.function
def train_step(images, labels):
  with tf.GradientTape() as tape:
    predictions = model(images, training=True)
    loss = loss_object(labels, predictions)
  gradients = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(gradients, model.trainable_variables))
 
  train_loss_metric.update_state(loss)
  train_accuracy_metric.update_state(labels, predictions)
 
@tf.function
def val_step(images, labels):
  predictions = model(images, training=False)
  val_accuracy_metric.update_state(labels, predictions)

In [ ]:
def train_model(model, train_ds, val_ds, epochs=10):
  for epoch in range(epochs):
    start = time.time()
    print(f"\nEpoch {epoch+1}/{epochs}")
 
    # Reset metrics di awal setiap epoch
    train_loss_metric.reset_state()
    train_accuracy_metric.reset_state()
    val_accuracy_metric.reset_state()
 
    for images, labels in train_ds:
      train_step(images, labels)
 
    for val_images, val_labels in val_ds:
      val_step(val_images, val_labels)
 
    print(
        f"Train Accuracy: {train_accuracy_metric.result() * 100:.2f}%, ",
        f"Val Accuracy: {val_accuracy_metric.result() * 100:.2f}%",
        f"({time.time() - start:.2f}s)")

In [ ]:
model = MyModel()
train_model(model, train_ds, val_ds)

In [ ]:
def dynamic_train_model(model, train_ds, val_ds, epochs=10, stagnation_threshold=3):
  best_val_acc = 0
  stagnation_counter = 0
 
  for epoch in range(epochs):
    start = time.time()
    print(f"\nEpoch {epoch+1}/{epochs}")
 
    train_loss_metric.reset_state()
    train_accuracy_metric.reset_state()
    val_accuracy_metric.reset_state()
 
    for images, labels in train_ds:
      train_step(images, labels)
 
    for val_images, val_labels in val_ds:
      val_step(val_images, val_labels)
        
    print(
        f"Train Accuracy: {train_accuracy_metric.result() * 100:.2f}%, ",
        f"Val Accuracy: {val_accuracy_metric.result() * 100:.2f}%",
        f"({time.time() - start:.2f}s)")
 
    # Deteksi stagnasi
    val_acc = val_accuracy_metric.result().numpy()
 
    if val_acc > best_val_acc:
      best_val_acc = val_acc
      stagnation_counter = 0
    else:
      stagnation_counter += 1
 
    if stagnation_counter >= stagnation_threshold:
      print("Performa stagnan, training dihentikan lebih awal!")
      break

In [ ]:
model = MyModel()
dynamic_train_model(model, train_ds, val_ds)